In [2]:
import pandas as pd
import os

In [ ]:
def clean_orders(orders):
    if orders is None or orders.empty:
        raise ValueError("Table is empty or non-existant")
    
    initial_rows = orders.shape[0]
    
    expected_columns = ["order_id", "customer_id", "product_id", "order_date", "quantity", "unit_price", "status"]
    existing_columns = set(orders.columns)
    not_present = []
    
    for col in expected_columns:    
        if col not in existing_columns:
            not_present.append(col)
    
    if len(not_present) >= 1:
        raise KeyError(f"Missing following columns from table: {not_present}")
    
    orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
    orders = orders.dropna(subset=["order_date"])
    
    orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")
    orders = orders[orders["quantity"] > 0]
    
    orders["unit_price"] = pd.to_numeric(orders["unit_price"], errors="coerce")
    orders = orders[orders["unit_price"] > 0]
    
    orders["status"] = orders["status"].str.lower().str.strip()
    valid_statuses = ["completed", "cancelled", "returned"]
    orders = orders[orders["status"].isin(valid_statuses)]
    
    orders["order_id"] = orders["order_id"].astype(str)
    orders["customer_id"] = orders["customer_id"].astype(str)
    orders["product_id"] = orders["product_id"].astype(str)
    
    orders = orders.dropna(subset=["order_id", "customer_id", "product_id"])
    orders = orders.drop_duplicates(subset=["order_id"])
    
    final_rows = orders.shape[0]
    
    print(f"Removed {abs(final_rows - initial_rows)} from the table")